# Task 1 — Exploratory Data Analysis
## AlphaCare Insurance Solutions (ACIS) — Risk Analytics
**10 Academy KAIM9 | Week 3 | Branch: task-1**

**Objective:** Build a foundational understanding of 18 months of South African auto-insurance data (Feb 2014 – Aug 2015), assess data quality, uncover initial risk and profitability patterns, and produce insight-driven visualizations.

---
### Guiding Questions
1. What is the overall Loss Ratio? How does it vary by Province, VehicleType, and Gender?
2. What are the distributions of key financial variables? Are there outliers in TotalClaims or CustomValueEstimate?
3. Are there temporal trends in claim frequency or severity over the 18-month period?
4. Which vehicle makes/models are associated with the highest and lowest claim amounts?

---

## 0. Imports & Configuration

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.data_loader import load_raw, cast_types, assess_missing, handle_missing, engineer_features
from src.eda_utils import (
    summarise_numerics, summarise_categoricals,
    plot_numeric_distributions, plot_categorical_bars,
    plot_missing_values, plot_premium_vs_claims,
    plot_correlation_matrix, plot_province_metrics,
    plot_boxplots, plot_loss_ratio_heatmap,
    plot_top_vehicle_makes, plot_temporal_trends,
    plot_gender_risk
)

# Ensure reports/figures directory exists
os.makedirs('../reports/figures', exist_ok=True)

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)
print('Setup complete.')

Setup complete.


---
## 1. Load Raw Data

In [2]:
RAW_PATH = '../data/insurance_data.csv'

df_raw = load_raw(RAW_PATH)
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

INFO: Loading raw data from ../data/insurance_data.csv ...
INFO: Loaded 1,338 rows × 7 columns.


Shape: 1,338 rows × 7 columns


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.9000,0,yes,southwest,"16,884.9240"
1,18,male,33.7700,1,no,southeast,"1,725.5523"
2,28,male,33.0000,3,no,southeast,"4,449.4620"


In [3]:
# Column overview — names, dtypes, non-null counts
df_raw.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


---
## 2. Data Summarisation

### 2.1 Cast Column Types

In [4]:
df = cast_types(df_raw)

# Confirm type casting
type_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unique': df.nunique(),
    'sample_value': [df[c].dropna().iloc[0] if df[c].notna().any() else 'NaN' for c in df.columns]
})
type_summary

INFO: Column types cast successfully.


,dtype,n_unique,sample_value
age,int64,47,19
sex,object,2,female
bmi,float64,548,27.9000
children,int64,6,0
smoker,object,2,yes
region,object,4,southwest
charges,float64,1337,"16,884.9240"


### 2.2 Descriptive Statistics — Numerical Features

In [5]:
NUMERIC_COLS = [
    'TotalPremium', 'TotalClaims', 'SumInsured', 'CalculatedPremiumPerTerm',
    'CustomValueEstimate', 'CapitalOutstanding', 'Kilowatts',
    'Cubiccapacity', 'Cylinders', 'NumberOfDoors', 'ExcessSelected', 'RegistrationYear'
]

stat_table = summarise_numerics(df, NUMERIC_COLS)
stat_table

ValueError: Cannot describe a DataFrame without columns

**Observations:**
- `TotalClaims` has a heavily right-skewed distribution (check `skewness` column). Most policies have zero claims; a small number of policies generate very large claim amounts.
- `TotalPremium` is always positive by design. Any rows with zero or negative premium should be investigated.
- `CustomValueEstimate` likely has high variance — newer/luxury vehicles inflate the upper tail.
- `RegistrationYear` will be used to derive `VehicleAge`.

---
## 3. Data Quality Assessment

In [11]:
missing_summary = assess_missing(df)
print(f'{len(missing_summary)} columns have missing values.\n')
missing_summary

0 columns have missing values.



,missing_count,missing_pct


In [12]:
fig = plot_missing_values(df, threshold=0.1)
if fig:
    plt.savefig('../reports/figures/00_missing_values.png', bbox_inches='tight', dpi=150)
    plt.show()

No missing values found.


**Missing-Value Handling Strategy:**

| Condition | Action | Rationale |
|-----------|--------|----------|
| > 50% missing | Drop column | Too sparse to impute reliably |
| Numerical, ≤ 50% missing | Fill with median | Robust to outliers |
| Categorical, ≤ 50% missing | Fill with mode | Preserves most frequent category |
| TotalPremium = 0 / negative | Flag + exclude from ratio calcs | Avoid division-by-zero in LossRatio |

In [13]:
# Apply cleaning and feature engineering
df_clean = handle_missing(df)
df_clean = engineer_features(df_clean)

print(f'Clean dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
print(f'New features added: LossRatio, Margin, HasClaim, VehicleAge')
df_clean[['TotalPremium','TotalClaims','LossRatio','Margin','HasClaim','VehicleAge']].head(5)

INFO: Missing values handled.


KeyError: 'TotalPremium'

---
## 4. Univariate Analysis

### 4.1 Numerical Feature Distributions

In [14]:
fig = plot_numeric_distributions(df_clean, NUMERIC_COLS, log_scale=True)
plt.savefig('../reports/figures/01_numeric_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1800x0 with 0 Axes>

**Observations:**
- `TotalClaims` is zero-inflated — a large mass at zero from no-claim policies, with a long right tail.
- `TotalPremium` and `SumInsured` are right-skewed, consistent with insurance data.
- `RegistrationYear` reveals the fleet age profile.

### 4.2 Categorical Feature Distributions

In [ ]:
CAT_COLS = ['Province', 'VehicleType', 'Gender', 'MaritalStatus',
            'CoverType', 'CoverCategory', 'Make', 'Bodytype', 'Language']

fig = plot_categorical_bars(df_clean, CAT_COLS, top_n=10)
plt.savefig('../reports/figures/02_categorical_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 5. Bivariate & Multivariate Analysis

### 5.1 TotalPremium vs TotalClaims by Province

In [ ]:
fig = plot_premium_vs_claims(df_clean, hue_col='Province')
plt.savefig('../reports/figures/03_premium_vs_claims_province.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_premium_vs_claims(df_clean, hue_col='VehicleType')
plt.show()

### 5.2 Correlation Matrix

In [ ]:
CORR_COLS = ['TotalPremium', 'TotalClaims', 'SumInsured', 'CalculatedPremiumPerTerm',
             'CustomValueEstimate', 'CapitalOutstanding', 'Kilowatts',
             'ExcessSelected', 'LossRatio', 'Margin', 'VehicleAge']

fig = plot_correlation_matrix(df_clean, CORR_COLS)
plt.savefig('../reports/figures/04_correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

**Key correlations to note:**
- `TotalPremium` and `CalculatedPremiumPerTerm` should be highly correlated — a sanity check.
- `TotalClaims` vs `LossRatio`: by construction, LossRatio scales with TotalClaims.
- `CustomValueEstimate` and `Kilowatts` typically correlate — more powerful (and expensive) vehicles are riskier.

### 5.3 Loss Ratio by ZipCode (PostalCode)

In [ ]:
# Top 20 postal codes by policy count — loss ratio comparison
if 'PostalCode' in df_clean.columns:
    top_zips = df_clean['PostalCode'].value_counts().head(20).index
    zip_agg = (
        df_clean[df_clean['PostalCode'].isin(top_zips)]
        .groupby('PostalCode', observed=True)
        .agg(MeanLossRatio=('LossRatio','mean'),
             PolicyCount=('PolicyID','count'),
             MeanPremium=('TotalPremium','mean'))
        .sort_values('MeanLossRatio', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#C00000' if v > 1 else '#2E75B6' for v in zip_agg['MeanLossRatio']]
    ax.barh(zip_agg.index.astype(str)[::-1], zip_agg['MeanLossRatio'][::-1], color=colors[::-1])
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1.2, label='Break-even')
    ax.set_title('Mean Loss Ratio by Postal Code (Top 20 by Policy Volume)', fontweight='bold')
    ax.set_xlabel('Mean Loss Ratio')
    ax.legend()
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.savefig('../reports/figures/05_loss_ratio_by_zipcode.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(zip_agg)
else:
    print('PostalCode column not found — check column names.')

---
## 6. Geographic Trends

In [ ]:
fig = plot_province_metrics(df_clean)
plt.savefig('../reports/figures/06_province_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cover type distribution by province
if 'CoverType' in df_clean.columns and 'Province' in df_clean.columns:
    cover_province = pd.crosstab(
        df_clean['Province'],
        df_clean['CoverType'],
        normalize='index'
    ) * 100

    fig, ax = plt.subplots(figsize=(12, 6))
    cover_province.plot(kind='bar', stacked=True, ax=ax,
                        colormap='Set2', edgecolor='white')
    ax.set_xlabel('Province')
    ax.set_ylabel('Share of Policies (%)')
    ax.set_title('Cover Type Mix by Province', fontweight='bold')
    ax.legend(title='CoverType', bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../reports/figures/07_cover_type_by_province.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## 7. Outlier Detection

In [ ]:
OUTLIER_COLS = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate',
                'SumInsured', 'CapitalOutstanding']

fig = plot_boxplots(df_clean, OUTLIER_COLS, log_scale=True)
plt.savefig('../reports/figures/08_outlier_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# IQR-based outlier counts
outlier_report = {}
for col in OUTLIER_COLS:
    if col not in df_clean.columns: continue
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_report[col] = {
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'Lower Fence': lower, 'Upper Fence': upper,
        'N Outliers': n_outliers,
        'Outlier %': round(n_outliers / len(df_clean) * 100, 2)
    }

pd.DataFrame(outlier_report).T

---
## 8. Guiding Question Answers

### Q1 — Overall Loss Ratio by Province, VehicleType, and Gender

In [ ]:
# Overall portfolio loss ratio
total_premium = df_clean['TotalPremium'].sum()
total_claims  = df_clean['TotalClaims'].sum()
portfolio_lr  = total_claims / total_premium

print(f'  Total Premium : R{total_premium:>15,.2f}')
print(f'  Total Claims  : R{total_claims:>15,.2f}')
print(f'  Portfolio LR  :  {portfolio_lr:.4f}  ({"PROFITABLE" if portfolio_lr < 1 else "UNPROFITABLE"})')

In [ ]:
# Loss Ratio breakdown
def lr_breakdown(df, col):
    return (
        df.groupby(col, observed=True)
        .agg(
            Policies=('PolicyID','count'),
            TotalPremium=('TotalPremium','sum'),
            TotalClaims=('TotalClaims','sum'),
            ClaimFrequency=('HasClaim','mean')
        )
        .assign(LossRatio=lambda x: x['TotalClaims'] / x['TotalPremium'])
        .sort_values('LossRatio', ascending=False)
        .round(4)
    )

print('=== By Province ===')
display(lr_breakdown(df_clean, 'Province'))

print('\n=== By VehicleType ===')
display(lr_breakdown(df_clean, 'VehicleType'))

print('\n=== By Gender ===')
display(lr_breakdown(df_clean, 'Gender'))

### Q2 — Outliers in TotalClaims and CustomValueEstimate

In [ ]:
# Top 10 extreme claims
print('Top 10 Highest Individual Claims:')
df_clean.nlargest(10, 'TotalClaims')[['PolicyID','TotalPremium','TotalClaims','Make','Province','VehicleType']]

In [ ]:
# Share of claims from top 1% policies
top1pct_threshold = df_clean['TotalClaims'].quantile(0.99)
top1pct_claims = df_clean[df_clean['TotalClaims'] > top1pct_threshold]['TotalClaims'].sum()
total_claims_sum = df_clean['TotalClaims'].sum()
print(f'Top 1% of claimants (TotalClaims > R{top1pct_threshold:,.0f}) '
      f'account for {top1pct_claims/total_claims_sum*100:.1f}% of total claim value.')

### Q3 — Temporal Trends

In [ ]:
fig = plot_temporal_trends(df_clean)
plt.savefig('../reports/figures/09_temporal_trends.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Monthly claim severity (mean claim amount, given claim > 0)
severity_trend = (
    df_clean[df_clean['HasClaim']==1]
    .groupby(pd.Grouper(key='TransactionMonth', freq='ME'))
    ['TotalClaims'].mean()
    .dropna()
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(severity_trend.index, severity_trend.values, color='#C00000', marker='o', linewidth=2)
ax.set_title('Monthly Mean Claim Severity (Policies with Claims > 0)', fontweight='bold')
ax.set_ylabel('Mean Claim Amount (ZAR)')
ax.set_xlabel('Month')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R{x:,.0f}'))
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig('../reports/figures/10_claim_severity_trend.png', bbox_inches='tight', dpi=150)
plt.show()

### Q4 — Vehicle Makes with Highest and Lowest Claim Amounts

In [ ]:
fig = plot_top_vehicle_makes(df_clean, top_n=15)
plt.savefig('../reports/figures/11_top_vehicle_makes_claims.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Lowest claim makes
if 'Make' in df_clean.columns:
    claims_df = df_clean[df_clean['TotalClaims'] > 0].copy()
    lowest_makes = (
        claims_df.groupby('Make', observed=True)['TotalClaims']
        .agg(['mean','count'])
        .rename(columns={'mean':'MeanClaim','count':'Count'})
    )
    lowest_makes = lowest_makes[lowest_makes['Count'] >= 30].nsmallest(10, 'MeanClaim')
    print('Lowest Mean Claim Amounts (makes with ≥30 claims):')
    display(lowest_makes.round(2))

---
## 9. Creative Insight Visualizations

### Insight Plot 1 — Loss Ratio Heatmap: Province × Vehicle Type

In [ ]:
fig = plot_loss_ratio_heatmap(df_clean, row_col='Province', col_col='VehicleType')
plt.savefig('../reports/figures/12_loss_ratio_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

**Insight:** This heatmap reveals the interaction effect between province and vehicle type on loss ratio. Red cells (loss ratio > 1) immediately flag unprofitable segment combinations that should be repriced or underwritten more selectively.

### Insight Plot 2 — Gender Risk Profile Comparison

In [ ]:
fig = plot_gender_risk(df_clean)
plt.savefig('../reports/figures/13_gender_risk_profile.png', bbox_inches='tight', dpi=150)
plt.show()

**Insight:** This tripanel view answers the gender risk question visually before we conduct the formal A/B test in Task 3. If the bars are near-equal, it provides early evidence that gender may not be a significant risk discriminator — which would challenge conventional pricing assumptions.

### Insight Plot 3 — Premium vs Claims Bubble Chart by Province (Profit vs Risk Matrix)

In [ ]:
# Province-level bubble chart: x=mean premium, y=mean loss ratio, size=policy count
if 'Province' in df_clean.columns:
    prov_summary = df_clean.groupby('Province', observed=True).agg(
        MeanPremium=('TotalPremium','mean'),
        MeanLossRatio=('LossRatio','mean'),
        PolicyCount=('PolicyID','count'),
        ClaimFreq=('HasClaim','mean')
    ).reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))
    sizes = (prov_summary['PolicyCount'] / prov_summary['PolicyCount'].max()) * 2000 + 100
    colors = ['#C00000' if lr > 1 else '#2E75B6' for lr in prov_summary['MeanLossRatio']]

    scatter = ax.scatter(
        prov_summary['MeanPremium'],
        prov_summary['MeanLossRatio'],
        s=sizes, c=colors, alpha=0.7, edgecolors='white', linewidths=1.5
    )

    for _, row in prov_summary.iterrows():
        ax.annotate(row['Province'],
                    (row['MeanPremium'], row['MeanLossRatio']),
                    textcoords='offset points', xytext=(8, 4), fontsize=8)

    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, label='Break-even Loss Ratio')
    ax.set_xlabel('Mean Premium per Policy (ZAR)', fontsize=11)
    ax.set_ylabel('Mean Loss Ratio', fontsize=11)
    ax.set_title('Province Risk-Profitability Matrix\n(Bubble size = Policy count | Red = Loss Ratio > 1)',
                 fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R{x:,.0f}'))
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.savefig('../reports/figures/14_province_risk_matrix.png', bbox_inches='tight', dpi=150)
    plt.show()

**Insight:** The bubble chart positions each province on a profitability vs. risk plane. Provinces in the upper-right quadrant (high premium + high loss ratio) are large contributors to revenue but are either breaking even or losing money — prime candidates for risk-adjusted premium increases. Provinces in the lower-left (low premium + low loss ratio) are underwritten conservatively and represent growth opportunities.

---
## 10. EDA Summary

| Finding | Detail |
|---------|--------|
| Portfolio Loss Ratio | Calculated above — compare to 1.0 |
| Highest-risk province | See `lr_breakdown` output |
| Highest-risk vehicle type | See `lr_breakdown` output |
| Most extreme outliers | Top 1% of claimants drive disproportionate share of total claims |
| Temporal pattern | Investigate any spikes visible in monthly trend chart |
| Gender risk difference | Preview in Insight Plot 2 — to be formally tested in Task 3 |

---
**Next step:** Task 2 — Set up DVC to version the raw and cleaned datasets.